# Sujet 07 — Santander · Porteurs **P1** et **P4**
Ce carnet ne contient que les étapes de **P1** et **P4** du tableau. Les autres étapes (P2, P3) sont absentes, sauf une cellule de **prérequis** (en gris, ci-dessous) qui fournit juste ce dont P1/P4 ont besoin pour tourner.

| Porteur | Enchaînement | Dans ce carnet |
|---|---|---|
| **P1** | 1.1 Inventorier la table → 1.8 Fuites | 1.1, 1.8 |
| **P4** | 1.7 Doublons → 1.11 La métrique → 1.12 Le coût des erreurs | 1.7, 1.11, 1.12 |

L'exécution se fait dans l'ordre numérique : 1.1 → *(prérequis)* → 1.7 → 1.8 → 1.11 → 1.12.

**Avant de lancer** : *Add Input* → jeu de données avec `train.csv` et `a_decider.csv` (ou mettez-les à côté du carnet en local). Remplissez `NOM_EQUIPE` et `MEMBRES` dans la configuration.

## Configuration

In [ ]:
import os, glob, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
warnings.filterwarnings("ignore")

# ---------------- PARAMÈTRES ----------------
SEED         = 42      # graine principale
N_SPLITS     = 5       # plis de la validation croisée (modèles rapides)
N_SPLITS_HGB = 3       # plis pour les arbres (plus lents)
HOLDOUT_FRAC = 0.20    # part gardée intouchée, utilisée UNE seule fois
K_LIST       = [1, 5, 10, 50, 100, 200]   # courbe de sélection
ROBUST_SEEDS = [1, 2, 3]                  # graines du test de robustesse
COST_FN      = 10.0    # coût d'une vente manquée (hypothèse, à justifier)
COST_FP      = 1.0     # coût d'une sollicitation inutile (hypothèse)
PLANCHER_ENONCE = 0.5639
# --------------------------------------------

on_kaggle = os.path.exists("/kaggle")
OUT = "/kaggle/working/sorties" if on_kaggle else "sorties"
os.makedirs(OUT, exist_ok=True)

def find(name):
    cands = []
    for root in ["/kaggle/input", "."]:
        if not os.path.exists(root): continue
        for r, d, fs in os.walk(root):
            for f in fs:
                if f.lower() == name.lower():
                    cands.append(os.path.join(r, f))
    cands = [c for c in cands if os.path.getsize(c) > 0]
    if not cands:
        vus = [os.path.join(r, f) for r, d, fs in os.walk("/kaggle/input") for f in fs] if os.path.exists("/kaggle/input") else []
        raise FileNotFoundError(f"{name} introuvable. Fichiers vus dans /kaggle/input : {vus[:20] if vus else 'AUCUN (ajoutez votre jeu de données via Add Input > Your Work)'}")
    return max(cands, key=os.path.getsize)

T0 = time.time()
AUDIT = []
def audit(etape, porteur, titre, constat):
    AUDIT.append(dict(etape=etape, porteur=porteur, titre=titre, constat=constat))
    print(f"[{etape} · {porteur}] {constat}")

# ---------------- ÉQUIPE (à remplir) ----------------
NOM_EQUIPE = "Équipe XX"
MEMBRES    = ["Prénom 1", "Prénom 2", "Prénom 3"]      # <- mettez vos trois prénoms
PORTEURS   = {"P1": MEMBRES[0], "P4": MEMBRES[1]}   # qui tient P1, qui tient P4 : adaptez
COURS      = "ESGC — Sujet 07 Santander"
DATE       = pd.Timestamp.today().strftime("%d/%m/%Y")
RUN_PIPELINE = False    # True : exécute aussi pipeline.py de bout en bout (mesure du temps) ; ajoute quelques minutes
MUT = {}
BASE = "/kaggle/working" if on_kaggle else "."

In [ ]:
print(f"{COURS} — {NOM_EQUIPE}\nMembres : {', '.join(MEMBRES)}\nPorteurs : {PORTEURS}\nDate : {DATE}")

In [ ]:
# Diagnostic des données : à lire avant de continuer
vus = []
if os.path.exists("/kaggle/input"):
    for r, d, fs in os.walk("/kaggle/input"):
        for f in fs:
            p = os.path.join(r, f); vus.append((p, os.path.getsize(p)))
for p, t in vus: print(f"{t/1e6:9.1f} Mo  {p}")
if not vus:
    print("AUCUN fichier dans /kaggle/input : ajoutez votre jeu de données (Add Input > Your Work > le +) ou, en local, mettez les CSV à côté du carnet.")
elif max(t for _, t in vus) < 1_000_000:
    print("ATTENTION : tous les fichiers pèsent moins de 1 Mo. Ce ne sont pas les vrais fichiers (train.csv doit faire environ 250 Mo). Recréez le jeu de données.")

## Boîte à outils
Définitions seulement : rien n'est calculé ici.

In [ ]:
def uni_auc(X, y):
    # AUC de chaque colonne seule, vectorisée (rangs)
    r = rankdata(X, axis=0)
    n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum(axis=0) - n1*(n1+1)/2) / (n1*n0)

def score(model, X):
    # score continu : logit de la proba (sert à l'AUC et à la calibration)
    p = np.clip(model.predict_proba(X)[:, 1], 1e-9, 1-1e-9)
    return np.log(p/(1-p))

def cv_eval(model, X, y, n_splits=N_SPLITS, seed=SEED):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y)); aucs = []
    t = time.time()
    for tr, va in skf.split(X, y):
        m = clone(model).fit(X[tr], y[tr])
        oof[va] = score(m, X[va])
        aucs.append(roc_auc_score(y[va], oof[va]))
    return dict(oof=oof, mean=np.mean(aucs), std=np.std(aucs), time=time.time()-t)

JOURNAL = []
def log(nom, res, note="", k=None):
    JOURNAL.append(dict(essai=nom, k=k, auc=round(res["mean"], 5), ecart_type=round(res["std"], 5),
                        secondes=round(res["time"], 1), note=note))
    print(f"{nom:<38} AUC {res['mean']:.5f} ± {res['std']:.5f}  ({res['time']:.0f} s)  {note}")

In [ ]:
class TopK(BaseEstimator, TransformerMixin):
    # garde les k variables dont l'AUC seule est la plus éloignée de 0,5, calculée sur le pli d'entraînement
    def __init__(self, k=200): self.k = k
    def fit(self, X, y):
        a = uni_auc(np.asarray(X), np.asarray(y))
        self.idx_ = np.argsort(-np.abs(a - 0.5))[:self.k]
        return self
    def transform(self, X): return np.asarray(X)[:, self.idx_]

class NBBin(BaseEstimator, ClassifierMixin):
    # Naive Bayes par variable discrétisée en quantiles : somme de rapports de vraisemblance (modèle additif)
    def __init__(self, n_bins=20, alpha=1.0): self.n_bins = n_bins; self.alpha = alpha
    def fit(self, X, y):
        X = np.asarray(X); y = np.asarray(y); self.classes_ = np.array([0, 1])
        qs = np.linspace(0, 1, self.n_bins + 1)[1:-1]
        self.edges_ = [np.unique(np.quantile(X[:, j], qs)) for j in range(X.shape[1])]
        self.llr_ = []
        for j, e in enumerate(self.edges_):
            b = np.searchsorted(e, X[:, j]); nb = len(e) + 1
            c1 = np.bincount(b[y == 1], minlength=nb) + self.alpha
            c0 = np.bincount(b[y == 0], minlength=nb) + self.alpha
            self.llr_.append(np.log(c1/c1.sum()) - np.log(c0/c0.sum()))
        self.prior_ = np.log(y.mean()/(1 - y.mean()))
        return self
    def decision_function(self, X):
        X = np.asarray(X); s = np.full(len(X), self.prior_)
        for j, e in enumerate(self.edges_): s += self.llr_[j][np.searchsorted(e, X[:, j])]
        return s
    def predict_proba(self, X):
        p = 1/(1 + np.exp(-self.decision_function(X))); return np.c_[1 - p, p]

class FreqAdd(BaseEstimator, TransformerMixin):
    # ajoute, pour chaque variable, log(1 + nb d'apparitions de la valeur arrondie) ; comptage appris sur le pli
    def __init__(self, decimals=4): self.decimals = decimals
    def fit(self, X, y=None):
        X = np.asarray(X)
        self.counts_ = [pd.Series(np.round(X[:, j], self.decimals)).value_counts() for j in range(X.shape[1])]
        return self
    def transform(self, X):
        X = np.asarray(X)
        F = np.column_stack([np.log1p(pd.Series(np.round(X[:, j], self.decimals)).map(self.counts_[j]).fillna(0).values)
                            for j in range(X.shape[1])])
        return np.hstack([X, F])

def hgb(depth=3, lr=0.1, it=200, **kw):
    return HistGradientBoostingClassifier(max_depth=depth, learning_rate=lr, max_iter=it,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=SEED, **kw)
def logreg(C=0.1): return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=300))

## 1.1 Inventorier la table · *P1*
**Question de l'oral :** « Qu'avez-vous reçu, exactement ? » Quels fichiers, quel poids, quelles colonnes, et pourquoi `verite.csv` ne doit pas être là.

In [ ]:
FILES = {n: find(n) for n in ["train.csv", "a_decider.csv"]}
LIGNES = {}
for n, p in FILES.items():
    if os.path.getsize(p) < 1_000_000:
        raise RuntimeError(f"{n} ne pèse que {os.path.getsize(p)} octets : ce n'est pas le vrai fichier. "
                           "Retéléchargez le dossier du Drive du cours et recréez le jeu de données Kaggle.")
    with open(p, "rb") as fh: LIGNES[n] = sum(1 for _ in fh) - 1
for n, p in FILES.items():
    head = pd.read_csv(p, nrows=5)
    print(f"{n:<14} {os.path.getsize(p)/1e6:7.1f} Mo | {head.shape[1]} colonnes | {list(head.columns[:3])} ... {head.columns[-1]}")
cols_tr = pd.read_csv(FILES["train.csv"], nrows=0).columns
cols_te = pd.read_csv(FILES["a_decider.csv"], nrows=0).columns
extra_tr = sorted(set(cols_tr) - set(cols_te)); extra_te = sorted(set(cols_te) - set(cols_tr))
print("Colonnes seulement dans train :", extra_tr, "| seulement dans a_decider :", extra_te)
verite = glob.glob("/kaggle/input/**/verite.csv", recursive=True) + glob.glob("./**/verite.csv", recursive=True)
print("verite.csv présent :", bool(verite), "(doit être False)")
print("Lignes : train %d (attendu 160000) | a_decider %d (attendu 40000) -> conforme : %s" % (
    LIGNES["train.csv"], LIGNES["a_decider.csv"], LIGNES["train.csv"] == 160000 and LIGNES["a_decider.csv"] == 40000))
inv = pd.DataFrame({"colonne": ["ID_code", "var_0 … var_199", "target"], "rôle": ["identifiant", "200 variables continues anonymisées", "cible 0/1 (train seulement)"]})
display(inv)
audit("1.1", "P1", "Inventaire", f"2 fichiers ; colonnes en plus dans train : {extra_tr} ; verite.csv présent : {bool(verite)}")

## Prérequis *(étapes de P2 et P3, version minimale)*
Chargement (1.2), découpage stratifié (1.9) et plancher (1.10) : seulement ce qu'il faut pour faire tourner 1.7, 1.8, 1.11 et 1.12. **À ne pas présenter comme travail de P1 ou P4** : les versions complètes (avec leurs contrôles) sont dans le carnet commun.

In [ ]:
try:
    FILES
except NameError:
    FILES = {}
for n in ["train.csv", "a_decider.csv"]:
    if n not in FILES: FILES[n] = find(n)
    taille = os.path.getsize(FILES[n])
    print(f"{n:<14} {FILES[n]}  ({taille/1e6:.1f} Mo)")
    if taille < 1_000_000:
        raise RuntimeError(f"{n} ne pèse que {taille} octets : ce n'est pas le vrai fichier. Recréez le jeu de données.")
train = pd.read_csv(FILES["train.csv"]); test = pd.read_csv(FILES["a_decider.csv"])
FEATS = [c for c in train.columns if c.startswith("var_")]
X_all = train[FEATS].values.astype(np.float64); y_all = train["target"].values.astype(int)
X_dev, X_hold, y_dev, y_hold = train_test_split(X_all, y_all, test_size=HOLDOUT_FRAC, stratify=y_all, random_state=SEED)
a = uni_auc(X_dev, y_dev)
ranking = pd.DataFrame({"variable": FEATS, "auc_seule": np.maximum(a, 1 - a), "sens": np.where(a >= 0.5, "+", "-")}).sort_values("auc_seule", ascending=False)
PLANCHER = ranking.auc_seule.iloc[0]
print("train", train.shape, "| a_decider", test.shape, "| dev", X_dev.shape, "| holdout", X_hold.shape)
print("Plancher (meilleure variable seule) : %s, AUC %.4f (énoncé : %s)" % (ranking.variable.iloc[0], PLANCHER, PLANCHER_ENONCE))

## 1.7 Doublons · *P4*
**Question de l'oral :** « Y a-t-il des clients en double, et des clients de `a_decider` déjà vus dans `train` ? » Ce dernier cas serait une fuite directe.

In [ ]:
d_id_tr = int(train.ID_code.duplicated().sum()); d_id_te = int(test.ID_code.duplicated().sum())
common_id = len(set(train.ID_code) & set(test.ID_code))
d_rows = int(train[FEATS].duplicated().sum())
h_tr = pd.util.hash_pandas_object(train[FEATS], index=False).values
h_te = pd.util.hash_pandas_object(test[FEATS], index=False).values
test_in_train = int(np.isin(h_te, h_tr).sum())
print("ID dupliqués : train %d | a_decider %d | communs aux deux : %d" % (d_id_tr, d_id_te, common_id))
print("Lignes dupliquées (hors ID) dans train : %d | lignes de a_decider déjà dans train : %d" % (d_rows, test_in_train))
audit("1.7", "P4", "Doublons", f"ID dupliqués {d_id_tr}/{d_id_te}, ID communs {common_id}, lignes dupliquées {d_rows}, test déjà dans train {test_in_train}")

## 1.8 Fuites · *P1*
**Question de l'oral :** « Comment savez-vous qu'aucune information interdite n'entre dans le modèle ? » Trois tests : l'identifiant, la variable « trop belle », la validation adversariale (train contre a_decider). Règle du cours : *une information inconnue au moment de la décision ne peut pas entrer dans le modèle.*

In [ ]:
for nom in ["train", "test", "FEATS", "uni_auc", "logreg", "audit"]:
    if nom not in globals():
        raise RuntimeError(f"'{nom}' n'existe pas : faites Run > Restart session puis Run All depuis le haut.")
from sklearn.model_selection import cross_val_score
# 1. l'ordre / le numéro de l'identifiant porte-t-il la cible ?
num_id = pd.to_numeric(train.ID_code.astype(str).str.extract(r"(\d+)")[0], errors="coerce").fillna(0).values
auc_id = roc_auc_score(train.target, num_id); auc_id = max(auc_id, 1 - auc_id)
# 2. une variable « trop belle » ?
a_all = uni_auc(train[FEATS].values, train.target.values); best = np.max(np.abs(a_all - 0.5)) + 0.5
# 3. validation adversariale train vs a_decider
n = min(20000, len(test))
Xa = np.vstack([train[FEATS].sample(n, random_state=SEED).values, test[FEATS].sample(n, random_state=SEED).values])
ya = np.r_[np.zeros(n), np.ones(n)]
auc_adv = cross_val_score(logreg(), Xa, ya, cv=3, scoring="roc_auc").mean()
print("AUC de l'ID_code seul : %.4f (≈0,5 attendu)" % auc_id)
print("Meilleure AUC d'une variable seule : %.4f (>0,9 serait suspect)" % best)
print("AUC adversariale train vs a_decider : %.4f (≈0,5 = mêmes distributions)" % auc_adv)
print("« target » absente de a_decider :", "target" not in test.columns)
audit("1.8", "P1", "Fuites", f"AUC(ID_code) {auc_id:.3f} ; meilleure variable seule {best:.3f} ; AUC adversariale {auc_adv:.3f}")

## 1.11 La métrique · *P4*
**Question de l'oral :** « Pourquoi l'AUC et pas l'exactitude ? » La cellule le démontre : répondre « non » à tout donne une exactitude élevée et zéro positif trouvé.

In [ ]:
acc0 = 1 - y_dev.mean()
auc_alea = roc_auc_score(y_dev, np.random.RandomState(SEED).rand(len(y_dev)))
j = FEATS.index(ranking.variable.iloc[0]); v = X_dev[:, j]
print("Exactitude en répondant « non » à tous : %.2f %% -> trompeuse (aucun positif trouvé)" % (100 * acc0))
print("AUC d'un score aléatoire : %.4f | AUC du plancher : %.4f" % (auc_alea, PLANCHER))
print("AUC invariante aux transformations croissantes : %.5f = %.5f" % (roc_auc_score(y_dev, v), roc_auc_score(y_dev, 3 * v + 1)))
print("""
L'AUC récompense : classer un positif au-dessus d'un négatif (probabilité qu'un positif tiré au hasard ait un score supérieur).
L'AUC ignore   : le seuil, la calibration des probabilités, et le coût relatif des erreurs.
-> on sépare le CLASSEMENT (jugé à l'AUC) de la DÉCISION (qui demande un seuil, voir 1.12).""")
audit("1.11", "P4", "Métrique", f"AUC ; exactitude triviale {100*acc0:.1f} % ; AUC aléatoire {auc_alea:.3f}")

## 1.12 Le coût des erreurs · *P4*
**Question de l'oral :** « Combien coûte une vente manquée contre une sollicitation inutile, et d'où viennent vos chiffres ? » Les valeurs `COST_FN` et `COST_FP` de la configuration sont des **hypothèses à justifier**.

In [ ]:
pi = y_dev.mean()
print("Coût moyen / client : ne solliciter personne = %.3f | solliciter tout le monde = %.3f" % (COST_FN * pi, COST_FP * (1 - pi)))
print("Seuil théorique : %.3f" % (COST_FP / (COST_FP + COST_FN)))
for ratio in [2, 5, 10, 20]:
    print(f"  si une vente manquée coûte {ratio}× une sollicitation inutile -> seuil {1/(1+ratio):.3f}")

# Coût de la règle plancher : on sollicite les q % les mieux classés sur la meilleure variable seule
sgn = 1 if ranking.sens.iloc[0] == "+" else -1
order = np.argsort(-sgn * v); yy = y_dev[order]
tp = np.cumsum(yy); fp = np.cumsum(1 - yy); n = len(yy)
cost_q = (COST_FN * (y_dev.sum() - tp) + COST_FP * fp) / n
q_best = (np.argmin(cost_q) + 1) / n
COUT_PLANCHER = cost_q.min()
print("Plancher : coût minimal %.3f en sollicitant %.1f %% des clients" % (COUT_PLANCHER, 100 * q_best))
audit("1.12", "P4", "Coût des erreurs", f"FN={COST_FN}, FP={COST_FP} ; seuil théorique {COST_FP/(COST_FP+COST_FN):.3f} ; coût du plancher {COUT_PLANCHER:.3f}/client")

## Récapitulatif P1 et P4

In [ ]:
A = pd.DataFrame(AUDIT)
A["_k"] = A.etape.str.split(".").str[1].astype(int); A = A.sort_values("_k").drop(columns="_k")
A.insert(2, "responsable", A.porteur.map(PORTEURS))
pd.set_option("display.max_colwidth", 170); display(A)
with open(f"{OUT}/audit_P1_P4.md", "w") as f:
    f.write(f"# Audit P1 et P4 — {NOM_EQUIPE} ({DATE})\n\n| étape | porteur | responsable | titre | constat |\n|---|---|---|---|---|\n")
    for _, r in A.iterrows(): f.write(f"| {r.etape} | {r.porteur} | {r.responsable} | {r.titre} | {r.constat} |\n")
print("Écrit :", f"{OUT}/audit_P1_P4.md")

## Fichier de résultats P1 et P4
Écrit dans `sorties/resultats_P1_P4.md` (avec les chiffres de CE run).

In [ ]:
manquants = [n for n in ["auc_id", "best", "auc_adv", "d_id_tr", "d_id_te", "common_id", "d_rows", "test_in_train",
                         "acc0", "auc_alea", "ranking", "PLANCHER", "COUT_PLANCHER", "q_best", "pi", "LIGNES",
                         "FILES", "extra_tr", "extra_te", "verite", "test"] if n not in globals()]
if manquants:
    raise RuntimeError("Étapes non exécutées. Variables manquantes : " + ", ".join(manquants) +
                       ". Faites Run > Restart session, puis Run All depuis le haut.")
sain_id  = auc_id < 0.52
sain_var = best < 0.90
sain_adv = abs(auc_adv - 0.5) < 0.02
verdict_fuite = ("Aucune fuite détectée par ces trois tests." if (sain_id and sain_var and sain_adv)
                 else "À EXAMINER : " + "; ".join(m for ok, m in [(sain_id, "l'identifiant semble informatif"), (sain_var, "une variable seule est suspecte"), (sain_adv, "train et a_decider se distinguent")] if not ok))
verdict_dbl = ("Aucun doublon ni recouvrement train/a_decider." if (d_id_tr + d_id_te + common_id + d_rows + test_in_train) == 0
               else f"À EXAMINER : {d_id_tr} ID dupliqués (train), {d_id_te} (a_decider), {common_id} ID communs, {d_rows} lignes dupliquées, {test_in_train} lignes de a_decider déjà dans train.")
files_tbl = "\n".join(f"| `{n}` | {os.path.getsize(p)/1e6:.1f} Mo | {LIGNES[n]} |" for n, p in FILES.items())
ratios = "\n".join(f"| {r}× | {1/(1+r):.3f} |" for r in [2, 5, 10, 20])
M = MEMBRES

R = f"""# Résultats des porteurs P1 et P4 — {COURS}
**{NOM_EQUIPE}** · P1 : {PORTEURS['P1']} · P4 : {PORTEURS['P4']} · {DATE}

---
# P1 — {PORTEURS['P1']}

## 1.1 Inventorier la table
| Fichier | Poids | Lignes |
|---|---|---|
{files_tbl}

- Colonnes présentes seulement dans `train.csv` : {extra_tr} (attendu : `target`). Seulement dans `a_decider.csv` : {extra_te}.
- `verite.csv` présent dans les données : **{bool(verite)}** (doit être non).
- Rôles : `ID_code` (identifiant interne), `var_0`…`var_199` (200 variables continues anonymisées), `target` (cible 0/1, train seulement).

## 1.8 Fuites
| Test | Résultat | Seuil d'alerte |
|---|---|---|
| AUC de l'`ID_code` seul | {auc_id:.4f} | proche de 0,5 attendu |
| Meilleure AUC d'une variable seule | {best:.4f} | > 0,9 suspect |
| AUC adversariale train contre a_decider | {auc_adv:.4f} | proche de 0,5 = mêmes distributions |
| `target` absente de `a_decider.csv` | {"target" not in test.columns} | doit être vrai |

**Conclusion : {verdict_fuite}**

---
# P4 — {PORTEURS['P4']}

## 1.7 Doublons
- ID dupliqués : train {d_id_tr}, a_decider {d_id_te} ; ID communs aux deux : {common_id}.
- Lignes dupliquées (hors ID) dans train : {d_rows} ; lignes de a_decider déjà présentes dans train : {test_in_train}.

**Conclusion : {verdict_dbl}**

## 1.11 La métrique
- Exactitude en répondant « non » à tous : **{100*acc0:.2f} %**, sans trouver aucun positif.
- AUC d'un score aléatoire : {auc_alea:.4f} ; AUC du plancher (meilleure variable seule, `{ranking.variable.iloc[0]}`) : {PLANCHER:.4f} (énoncé : {PLANCHER_ENONCE}).
- L'AUC récompense le bon classement (un positif au-dessus d'un négatif) ; elle ignore le seuil, la calibration et le coût des erreurs. Le classement se juge à l'AUC, la décision demande un seuil (1.12).

## 1.12 Le coût des erreurs
Hypothèses de l'équipe : vente manquée = **{COST_FN}**, sollicitation inutile = **{COST_FP}** *(à justifier par écrit : d'où viennent ces deux chiffres ?)*.

- Coût moyen par client : ne solliciter personne = {COST_FN*pi:.3f} ; solliciter tout le monde = {COST_FP*(1-pi):.3f}.
- Seuil théorique sur une probabilité calibrée : **{COST_FP/(COST_FP+COST_FN):.3f}**.
- Règle plancher : coût minimal **{COUT_PLANCHER:.3f}** par client, en sollicitant {100*q_best:.1f} % des clients.

| Une vente manquée coûte… | Seuil |
|---|---|
{ratios}

---
## Reproductibilité
Graine {SEED}, jeu de validation {int(HOLDOUT_FRAC*100)} % stratifié (prérequis P2/P3, version minimale). Temps d'exécution de ce carnet : {(time.time()-T0)/60:.1f} min.

## Transparence
Outils d'aide utilisés pour produire ce carnet : **à déclarer ici selon les règles du cours** (nature de l'aide, ce que l'équipe a vérifié et compris).
"""
open(f"{OUT}/resultats_P1_P4.md", "w").write(R)
print(R)